5. Plot a bubble chart to analyze the relationship between app size (in MB) and average rating,
with the bubble size representing the number of installs. Include a 
filter to show only apps with a rating higher than 3.5 and that belong to the 
"Games" category and installs should be more than 50k as well as this graph should
work only between 5 PM IST to 7 PM IST apart from that time we should not show this graph in dashboard itself.


In [ ]:
import pandas as pd

# Load the dataset to inspect its structure
data = pd.read_csv('Play Store Data.csv')

# Display the first few rows of the dataset to understand its structure
data.head()

In [ ]:
import numpy as np

# Data cleaning: Convert 'Size' to numeric (in MB)
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024  # Convert kB to MB
    else:
        return np.nan  # Handle 'Varies with device'

data['Size_MB'] = data['Size'].apply(convert_size)

In [ ]:
# Data cleaning: Convert 'Installs' to numeric
#data['Installs'] = data['Installs'].str.replace('[+,]', '', regex=True).astype(float)

# Step 1: Replace non-numeric values with '0' and convert to string for safe regex operations
data['Installs'] = data['Installs'].fillna('0').astype(str)

# Step 2: Remove non-numeric characters (commas, plus signs)
data['Installs'] = data['Installs'].replace(r'[\+,]', '', regex=True)

# Step 3: Convert to integers
data['Installs'] = pd.to_numeric(data['Installs'], errors='coerce').fillna(0).astype(int)



In [ ]:
# Filter data based on the given conditions
filtered_data = data[
    (data['Rating'] > 3.5) &
    (data['Category'] == 'GAME') &
    (data['Installs'] > 50000)
]

# Display the first few rows of the filtered dataset to verify
filtered_data.head()


In [ ]:
data.info()

In [ ]:
import pytz
from datetime import datetime

# Define function to check if the current time is between 5 PM to 7 PM IST
def is_within_time_window():
    # Get current time in IST
    ist = pytz.timezone('Asia/Kolkata')
    current_time = datetime.now(ist)
    # Define start and end times
    start_time = current_time.replace(hour=17, minute=0, second=0, microsecond=0)
    end_time = current_time.replace(hour=19, minute=0, second=0, microsecond=0)
    return start_time <= current_time <= end_time

# Check if current time is within the specified range
if is_within_time_window():
    # Plotting the bubble chart
    plt.figure(figsize=(12, 8))

    # Define bubble sizes scaled by the number of installs
    bubble_size = filtered_data['Installs'] / 1e6  # Scale down for visualization

    # Create the bubble chart
    plt.scatter(
        filtered_data['Size_MB'],  # X-axis: App size in MB
        filtered_data['Rating'],   # Y-axis: Average rating
        s=bubble_size,             # Bubble size: Number of installs
        alpha=0.6,                 # Transparency for better visualization
        color='skyblue',
        edgecolors="w",
        linewidth=0.5
    )

    # Add labels and title
    plt.title('Bubble Chart: App Size vs. Rating (Games)', fontsize=16)
    plt.xlabel('App Size (MB)', fontsize=14)
    plt.ylabel('Average Rating', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.colorbar(label='Number of Installs (scaled)', orientation='vertical')
    plt.tight_layout()

    # Show the plot
    plt.show()
else:
    print("The chart is not available outside 5 PM to 7 PM IST.")
